<a href="https://colab.research.google.com/github/nirajkulkarni3605/ML_II/blob/main/ripper_cn2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:


import pandas as pd
import random




data = {
    "Outlook": [
        "Sunny", "Sunny", "Overcast", "Rain", "Rain",
        "Rain", "Overcast", "Sunny", "Sunny", "Rain",
        "Sunny", "Overcast", "Overcast", "Rain"
    ],

    "Temperature": [
        "Hot", "Hot", "Hot", "Mild", "Cool",
        "Cool", "Cool", "Mild", "Cool", "Mild",
        "Mild", "Mild", "Hot", "Mild"
    ],

    "Humidity": [
        "High", "High", "High", "High", "Normal",
        "Normal", "Normal", "High", "Normal", "Normal",
        "Normal", "High", "Normal", "High"
    ],

    "Windy": [
        "False", "True", "False", "False", "False",
        "True", "True", "False", "False", "False",
        "True", "True", "False", "True"
    ],

    "Play": [
        "No", "No", "Yes", "Yes", "Yes",
        "No", "Yes", "No", "Yes", "Yes",
        "Yes", "Yes", "Yes", "No"
    ]
}

df = pd.DataFrame(data)




print("=" * 70)
print("OUR CREATED DATASET")
print("=" * 70)

print(df.to_string(index=False))

print("\nDataset size:", df.shape)





def rule_covers(row, rule):
    """
    Check whether a row satisfies all conditions in a rule.
    """

    for attribute, value in rule.items():

        if row[attribute] != value:
            return False

    return True


def covered_examples(data, rule):
    """
    Return examples covered by a rule.
    """

    mask = []

    for _, row in data.iterrows():

        if rule_covers(row, rule):
            mask.append(True)
        else:
            mask.append(False)

    return data[mask]


def rule_accuracy(data, rule, target, positive_class):
    """
    Calculate accuracy/purity of a rule.
    """

    covered = covered_examples(data, rule)

    if len(covered) == 0:
        return 0

    correct = sum(
        covered[target] == positive_class
    )

    return correct / len(covered)


def rule_precision(data, rule, target, positive_class):
    """
    Calculate precision of a rule.
    """

    covered = covered_examples(data, rule)

    if len(covered) == 0:
        return 0

    positives = sum(
        covered[target] == positive_class
    )

    return positives / len(covered)


def find_best_rule(
    data,
    target,
    positive_class
):
    """
    Find a good rule using a simple CN2/RIPPER-style
    sequential covering strategy.
    """

    attributes = [
        col for col in data.columns
        if col != target
    ]

    current_rule = {}

    remaining_attributes = attributes.copy()

    while len(remaining_attributes) > 0:

        best_condition = None
        best_score = -1

        # Try every possible attribute=value condition
        for attribute in remaining_attributes:

            values = data[attribute].unique()

            for value in values:

                candidate_rule = current_rule.copy()

                candidate_rule[attribute] = value

                covered = covered_examples(
                    data,
                    candidate_rule
                )

                if len(covered) == 0:
                    continue

                # Number of positive examples
                positive_count = sum(
                    covered[target] == positive_class
                )

                # Precision
                precision = (
                    positive_count /
                    len(covered)
                )

                # Prefer rules with high precision
                # and reasonable coverage.
                score = (
                    precision *
                    (positive_count / len(data))
                )

                if score > best_score:

                    best_score = score
                    best_condition = (
                        attribute,
                        value
                    )

        if best_condition is None:
            break

        current_rule[
            best_condition[0]
        ] = best_condition[1]

        remaining_attributes.remove(
            best_condition[0]
        )

        # Stop when rule becomes pure
        covered = covered_examples(
            data,
            current_rule
        )

        if len(covered) > 0:

            precision = sum(
                covered[target] == positive_class
            ) / len(covered)

            if precision == 1.0:
                break

    return current_rule


def cn2_ripper(
    data,
    target,
    positive_class
):
    """
    Simplified CN2/RIPPER-style
    sequential covering algorithm.
    """

    remaining_data = data.copy()

    rules = []

    while len(remaining_data) > 0:

        # Stop if no positive examples remain
        positive_examples = remaining_data[
            remaining_data[target] == positive_class
        ]

        if len(positive_examples) == 0:
            break

        # Learn one rule
        rule = find_best_rule(
            remaining_data,
            target,
            positive_class
        )

        if len(rule) == 0:
            break

        # Determine covered examples
        covered = covered_examples(
            remaining_data,
            rule
        )

        if len(covered) == 0:
            break

        precision = rule_precision(
            remaining_data,
            rule,
            target,
            positive_class
        )

        # Store only reasonably pure rules
        if precision >= 0.70:

            rules.append({
                "conditions": rule,
                "class": positive_class,
                "precision": precision,
                "coverage": len(covered)
            })

        # Remove covered examples
        remaining_data = remaining_data.drop(
            covered.index
        )

    return rules



target = "Play"
positive_class = "Yes"

rules = cn2_ripper(
    df,
    target,
    positive_class
)




print("\n")
print("=" * 70)
print("CN2 / RIPPER-STYLE CLASSIFICATION RULES")
print("=" * 70)


def print_rule(rule):

    conditions = []

    for attribute, value in rule["conditions"].items():

        conditions.append(
            f"{attribute} = {value}"
        )

    condition_text = " AND ".join(
        conditions
    )

    print(
        f"IF {condition_text} "
        f"THEN {target} = {rule['class']}"
    )

    print(
        f"   Precision: "
        f"{rule['precision']:.2f}"
    )

    print(
        f"   Coverage: "
        f"{rule['coverage']}"
    )


for i, rule in enumerate(rules, 1):

    print(f"\nRule {i}:")
    print_rule(rule)






def foil_information_gain(
    p0,
    n0,
    p1,
    n1
):
    """
    Calculate FOIL information gain.

    p0 = positive examples before adding condition
    n0 = negative examples before adding condition
    p1 = positive examples after condition
    n1 = negative examples after condition
    """

    if p1 == 0:
        return 0

    if p0 == 0:
        return 0

    import math

    old_ratio = p0 / (p0 + n0)

    new_ratio = p1 / (p1 + n1)

    if old_ratio == 0:
        return 0

    gain = p1 * (
        math.log2(new_ratio)
        - math.log2(old_ratio)
    )

    return gain


def foil_learn_rule(
    data,
    target,
    positive_class
):
    """
    Learn a single FOIL rule.
    """

    attributes = [
        col for col in data.columns
        if col != target
    ]

    rule = {}

    positives = data[
        data[target] == positive_class
    ]

    negatives = data[
        data[target] != positive_class
    ]

    while len(negatives) > 0:

        best_condition = None
        best_gain = -1

        p0 = len(positives)
        n0 = len(negatives)

        for attribute in attributes:

            if attribute in rule:
                continue

            values = data[
                attribute
            ].unique()

            for value in values:

                # Filter positive examples
                positive_subset = positives[
                    positives[attribute] == value
                ]

                # Filter negative examples
                negative_subset = negatives[
                    negatives[attribute] == value
                ]

                p1 = len(
                    positive_subset
                )

                n1 = len(
                    negative_subset
                )

                if p1 == 0:
                    continue

                gain = foil_information_gain(
                    p0,
                    n0,
                    p1,
                    n1
                )

                if gain > best_gain:

                    best_gain = gain

                    best_condition = (
                        attribute,
                        value
                    )

        if best_condition is None:
            break

        # Add best literal to rule
        attribute, value = best_condition

        rule[attribute] = value

        # Update examples
        positives = positives[
            positives[attribute] == value
        ]

        negatives = negatives[
            negatives[attribute] == value
        ]

    return rule


def foil_algorithm(
    data,
    target,
    positive_class
):
    """
    Sequential FOIL algorithm.
    """

    remaining_data = data.copy()

    rules = []

    while True:

        positives = remaining_data[
            remaining_data[target] == positive_class
        ]

        if len(positives) == 0:
            break

        negatives = remaining_data[
            remaining_data[target] != positive_class
        ]

        if len(negatives) == 0:
            break

        rule = foil_learn_rule(
            remaining_data,
            target,
            positive_class
        )

        if len(rule) == 0:
            break

        covered = covered_examples(
            remaining_data,
            rule
        )

        if len(covered) == 0:
            break

        # Calculate rule precision
        correct = sum(
            covered[target] == positive_class
        )

        precision = (
            correct / len(covered)
        )

        rules.append({
            "conditions": rule,
            "class": positive_class,
            "precision": precision,
            "coverage": len(covered)
        })

        # Remove positive examples covered by rule
        positive_covered = covered[
            covered[target] == positive_class
        ]

        remaining_data = remaining_data.drop(
            positive_covered.index
        )

    return rules




foil_rules = foil_algorithm(
    df,
    target,
    positive_class
)




print("\n")
print("=" * 70)
print("FOIL FIRST-ORDER CLASSIFICATION RULES")
print("=" * 70)


for i, rule in enumerate(foil_rules, 1):

    print(f"\nFOIL Rule {i}:")

    conditions = []

    for attribute, value in rule[
        "conditions"
    ].items():

        conditions.append(
            f"{attribute}(X, {value})"
        )

    condition_text = ", ".join(
        conditions
    )

    print(
        f"Play(X, Yes) :- "
        f"{condition_text}."
    )

    print(
        f"Precision: "
        f"{rule['precision']:.2f}"
    )

    print(
        f"Coverage: "
        f"{rule['coverage']}"
    )




def predict_using_rules(
    row,
    rules,
    default_class="No"
):

    for rule in rules:

        if rule_covers(
            row,
            rule["conditions"]
        ):

            return rule["class"]

    return default_class



predictions = []

for _, row in df.iterrows():

    prediction = predict_using_rules(
        row,
        foil_rules,
        default_class="No"
    )

    predictions.append(prediction)


df["FOIL_Prediction"] = predictions




correct = sum(
    df["Play"] ==
    df["FOIL_Prediction"]
)

accuracy = (
    correct /
    len(df)
)


print("\n")
print("=" * 70)
print("FOIL CLASSIFICATION PERFORMANCE")
print("=" * 70)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)




print("\nActual vs Predicted:")

print(
    df[
        [
            "Outlook",
            "Temperature",
            "Humidity",
            "Windy",
            "Play",
            "FOIL_Prediction"
        ]
    ].to_string(index=False)
)




print("\n")
print("=" * 70)
print("SUMMARY")
print("=" * 70)

print("""
1. A synthetic supervised classification dataset was created.

2. A CN2/RIPPER-style sequential covering method was
   used to generate classification rules.

3. The FOIL algorithm was implemented using information
   gain to select first-order literals.

4. FOIL generated rules in the form:

       Play(X, Yes) :-
           Outlook(X, Overcast).

5. The learned FOIL rules were then used to classify
   the examples.

6. Classification accuracy was calculated.
""")


OUR CREATED DATASET
 Outlook Temperature Humidity Windy Play
   Sunny         Hot     High False   No
   Sunny         Hot     High  True   No
Overcast         Hot     High False  Yes
    Rain        Mild     High False  Yes
    Rain        Cool   Normal False  Yes
    Rain        Cool   Normal  True   No
Overcast        Cool   Normal  True  Yes
   Sunny        Mild     High False   No
   Sunny        Cool   Normal False  Yes
    Rain        Mild   Normal False  Yes
   Sunny        Mild   Normal  True  Yes
Overcast        Mild     High  True  Yes
Overcast         Hot   Normal False  Yes
    Rain        Mild     High  True   No

Dataset size: (14, 5)


CN2 / RIPPER-STYLE CLASSIFICATION RULES

Rule 1:
IF Humidity = Normal AND Windy = False THEN Play = Yes
   Precision: 1.00
   Coverage: 4

Rule 2:
IF Outlook = Overcast THEN Play = Yes
   Precision: 1.00
   Coverage: 3

Rule 3:
IF Temperature = Mild AND Humidity = Normal THEN Play = Yes
   Precision: 1.00
   Coverage: 1

Rule 4:
IF Outloo